# The Neuron, Perceptron, and XOR Problem

## 📚 Learning Objectives

By completing this notebook, you will:
- Understand linear separability and the XOR problem
- Use multi-layer models to solve XOR

## 🔗 Where this fits

**Builds on:** Unit 3, lesson 01 "Regression vs Classification" — a linear classifier, pushed until it breaks on XOR.

**Used later in:** Course 08 (AIAT 122) — Unit 1, where the multi-layer fix is built and trained from scratch.

---


## 🎯 The four-row table that stopped neural networks for fifteen years

In July 1958 the New York Times reported on a US Navy demonstration of Frank
Rosenblatt's **perceptron** and told its readers the Navy expected the device to
be "the embryo of an electronic computer that will be able to walk, talk, see,
write, reproduce itself and be conscious of its existence". Rosenblatt's machine
was real and its learning rule worked — you will implement it in Unit 4.

In **1969** Marvin Minsky and Seymour Papert published *Perceptrons*, and among
its results was one small, devastating example: a single-layer perceptron
**cannot compute XOR**. Not "does badly on" — *cannot*, provably, for any weights
at all. Four rows of a truth table. Funding and interest in neural networks
collapsed and did not properly recover until backpropagation was popularised by
Rumelhart, Hinton and Williams in **1986** — a gap of roughly fifteen years.

The mathematics was correct and the conclusion drawn from it was too broad: the
proof was about *single-layer* perceptrons, and the fix — a hidden layer — was
already conceivable. What was missing was a practical way to train the hidden
layer. This notebook walks straight into that history: it shows you the wall, and
then shows you the door.

### What goes wrong without understanding linear separability

You will spend a week tuning a model that provably cannot represent the answer.
Linear separability is the property that decides whether *any* single-neuron
model — perceptron, logistic regression, linear SVM — can solve your problem. If
your classes cannot be split by a straight line (a plane, a hyperplane), no
learning rate, no number of epochs and no random seed will get you there, and the
model will not tell you why it is stuck.


# The Neuron, Perceptron, and XOR Problem

**Unit:** Unit 3: AI Concepts, Terminology, and Application Domains Part 2  

## 📚 Learning Objectives

By completing this notebook, you will:
- Understand neuron structure and mathematics
- Learn activation functions (Sigmoid, ReLU)
- Understand perceptron limitations
- Solve the XOR problem using neural networks
- Implement XOR with Keras

---

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
# Anatomy of one artificial neuron: inputs are weighted, summed with a bias, and
# squashed by an activation function — the building block of every network in this course.
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

print("=== The Neuron and Perceptron ===")
print("\nNeuron Structure:")
print("  - Inputs (x₁, x₂, ..., xₙ)")
print("  - Weights (w₁, w₂, ..., wₙ)")
print("  - Bias (b)")
print("  - Activation function (σ)")
print("  - Output: σ(Σ(wᵢ·xᵢ) + b)")

print("\nActivation Functions:")
print("  - Sigmoid: σ(x) = 1/(1+e⁻ˣ)")
print("  - ReLU: f(x) = max(0, x)")
print("  - Tanh: tanh(x)")

=== The Neuron and Perceptron ===

Neuron Structure:
  - Inputs (x₁, x₂, ..., xₙ)
  - Weights (w₁, w₂, ..., wₙ)
  - Bias (b)
  - Activation function (σ)
  - Output: σ(Σ(wᵢ·xᵢ) + b)

Activation Functions:
  - Sigmoid: σ(x) = 1/(1+e⁻ˣ)
  - ReLU: f(x) = max(0, x)
  - Tanh: tanh(x)


## The XOR Problem


### 🧰 Bridge: three Keras words you are about to meet

The next cell calls `model.compile(...)` and `model.fit(...)`. Three new terms appear:

- **loss** — a single number measuring how wrong the model currently is (here: `binary_crossentropy`).
- **optimizer** — the algorithm that nudges the weights to shrink the loss (here: `Adam`).
- **epoch** — one full pass over the training data; `epochs=1000` means 1000 passes.

Treat them as black boxes for now — notebook `04_gradient_descent_loss_functions.ipynb`
opens the box and implements a loss function and an optimizer from scratch.


In [2]:
# The famous XOR problem: no single straight line separates its 1s from its 0s, so a
# lone perceptron must fail — and one hidden layer fixes it. This result shaped NN history.
import numpy as np
# XOR Truth Table
# A single perceptron CANNOT solve XOR (not linearly separable)
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])  # XOR output

print("=== XOR Problem ===")
print("Input | Output")
print("------|-------")
for i, (x, y) in enumerate(zip(X_xor, y_xor)):
    print(f"{x}  |   {y}")

print("\n❌ Single perceptron cannot solve XOR!")
print("✅ Need hidden layers (non-linearity)")

# Solve XOR with neural network (hidden layer)
# A minimal multi-layer network: 2 hidden sigmoid neurons give the model the non-linearity XOR needs.
model = Sequential([
    Dense(2, activation='sigmoid', input_shape=(2,)),  # Hidden layer
    Dense(1, activation='sigmoid')  # Output layer
])

model.compile(optimizer=Adam(learning_rate=0.1), loss='binary_crossentropy', metrics=['accuracy'])

# Train
# 1000 epochs on just 4 examples — tiny data, but the network must bend its decision boundary.
history = model.fit(X_xor, y_xor, epochs=1000, verbose=0)

# Test
# Check all four truth-table rows: the hidden layer lets the network get XOR right.
predictions = model.predict(X_xor)
predicted_classes = (predictions > 0.5).astype(int).flatten()

print("\n=== Neural Network Solution ===")
print("Input | Expected | Predicted | Correct")
print("------|----------|-----------|--------")
for i, (x, y_true, y_pred) in enumerate(zip(X_xor, y_xor, predicted_classes)):
    correct = "✓" if y_true == y_pred else "✗"
    print(f"{x}  |    {y_true}     |    {y_pred}     |   {correct}")

print(f"\nAccuracy: {np.mean(y_xor == predicted_classes):.0%}")

=== XOR Problem ===
Input | Output
------|-------
[0 0]  |   0
[0 1]  |   1
[1 0]  |   1
[1 1]  |   0

❌ Single perceptron cannot solve XOR!
✅ Need hidden layers (non-linearity)


/Users/abdullah/venvs/ai-diploma-tf/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step



=== Neural Network Solution ===
Input | Expected | Predicted | Correct
------|----------|-----------|--------
[0 0]  |    0     |    0     |   ✓
[0 1]  |    1     |    0     |   ✗
[1 0]  |    1     |    1     |   ✓
[1 1]  |    0     |    1     |   ✗

Accuracy: 50%


## 📊 Two failures, two different causes

The cell above tried XOR twice, changing the model class:

| model | can it represent XOR? | did it solve XOR? | printed result |
|---|---|---|---|
| Single perceptron | **no** — provably (Minsky & Papert, 1969) | no | flagged before training: "❌ Single perceptron cannot solve XOR" |
| Network with a hidden layer | **yes** | **no, on this run** | **50%** accuracy — 2 of 4 rows correct |

**The conclusion these two rows support — and it is the point of the notebook:**
those are *not the same failure*. The perceptron failed because the right answer
is not inside its hypothesis space; no training procedure could have rescued it.
The network failed with the right answer fully inside its reach, because gradient
descent from this particular random start never found it.

Look at what the network actually settled on: for inputs (0,0), (0,1), (1,0),
(1,1) it output `0, 0, 1, 1` — which is simply the value of the **first** input. That is a straight line. **A model with the capacity to bend
the boundary produced the perceptron's answer anyway**, which is exactly why
"my architecture can represent it" and "my training run found it" have to be
checked separately. Notebook 03 fixes the random seed and reaches 100% with the
same architecture, which settles which of the two problems this run had.


## 💬 Discuss

Read the printed result before answering. The neural network in the cell above
reached **50% accuracy** on XOR: `[0 0] → 0 ✓`, `[0 1] → 0 ✗`, `[1 0] → 1 ✓`,
`[1 1] → 1 ✗`. Two out of four — the same score as a coin flip.

1. That network *has* a hidden layer, so it is capable of representing XOR — and
   it still failed. The next notebook builds the same architecture with a **fixed
   random seed** and reaches 100%. **Is a model that solves the problem on some
   seeds and not others a solution?** Say what you would put in a report, and what
   you would demand from a colleague who showed you only the successful run.
2. Look at what it actually learned: it outputs the value of the *second* input.
   That is a straight line through the plane — the perceptron's answer, produced
   by a network that did not need to settle for it. Argue whether the useful
   lesson here is about XOR, about initialisation, or about how easy it is to
   report a broken run as a working one.
3. Minsky and Papert's proof was right and the field over-generalised from it for
   fifteen years. Name a current claim of the form "neural networks cannot X".
   What evidence would you want before believing it is a property of the method
   rather than of the experiments run so far?


## ⚠️ Where this breaks

**This notebook's own output is the limits section.** The network scored **50%**.
Do not read past that.

- **Small networks on tiny datasets are unstable.** Four training rows, three
  hidden units, ReLU activations and no fixed seed: a fraction of initialisations
  put every hidden unit into the flat, zero-gradient region of ReLU ("dead
  units"), and once there, gradient descent has nothing to descend. The run above
  is one of those. **This is why notebook 03 sets a seed and says so in a comment**
  — not to make the demo look good, but because a reproducible failure is
  debuggable and an irreproducible one is not.
- **Reporting a single run is not reporting a result.** With this much variance,
  the honest report is "accuracy over N seeds: min, median, max", not one number.
  Any comparison of two models where the seed-to-seed spread exceeds the gap
  between them is a comparison of noise.
- **A perceptron cannot solve XOR, and this is not a fixable weakness.** It is a
  statement about what a single linear boundary can represent. The same limit
  applies to logistic regression and to a linear SVM on the same features.
  **Cheaper alternative before reaching for a network:** engineer a feature. Add
  the product `x1 * x2` as a third column and XOR becomes linearly separable — no
  hidden layer needed. Feature engineering and depth solve the same problem, and
  the feature is often cheaper and always more explainable.
- **XOR is not a benchmark.** Four rows, no test set, and train and test are the
  same data. It is a *proof by demonstration* of a representational limit, and it
  says nothing about how any model will behave on real data.
- **`accuracy_score` on 4 samples moves in 25% steps.** There is no meaningful
  difference between 50% and 75% here. Metrics need enough data to have resolution.


## 📚 References

1. Rosenblatt, F. (1958). *The Perceptron: A Probabilistic Model for Information Storage and Organization in the Brain*. Psychological Review, 65(6), 386–408.
2. Minsky, M., & Papert, S. (1969). *Perceptrons: An Introduction to Computational Geometry*. MIT Press.
3. Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986). *Learning Representations by Back-Propagating Errors*. Nature, 323, 533–536.
4. Liu, Z., Wang, Y., Vaidya, S., et al. (2024). *KAN: Kolmogorov-Arnold Networks*. arXiv. <https://arxiv.org/abs/2404.19756>